In [13]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import scipy.stats as stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

In [14]:
!pip install pymannkendall

In [15]:
df_clean = pd.read_csv('../data/data_clean.csv')
print(df_clean.columns)
df_labels = df_clean.copy()[['class_primary','class_secondary','attack']]
df_names = df_clean['english_name']
df_data = df_clean.drop(columns=['english_name','class_primary','class_secondary','attack'])

Index(['gen', 'english_name', 'percent_male', 'percent_female', 'height_m',
       'weight_kg', 'capture_rate', 'base_egg_steps', 'hp', 'attack',
       'defense', 'sp_attack', 'sp_defense', 'speed', 'against_normal',
       'against_fire', 'against_water', 'against_electric', 'against_grass',
       'against_ice', 'against_fighting', 'against_poison', 'against_ground',
       'against_flying', 'against_psychict', 'against_bug', 'against_rock',
       'against_ghost', 'against_dragon', 'against_dark', 'against_steel',
       'against_fairy', 'votes_first', 'votes_top_6', 'num_abilities',
       'evo_length', 'has_mega_evolution', 'has_gigantamax', 'rarity',
       'class_primary', 'class_secondary'],
      dtype='object')


In [16]:
import pymannkendall as mk

# Prepare dataframe: drop 'gen', add 'attack'
df_mk = df_data.drop(columns=['gen']).copy()
df_mk['attack'] = df_labels['attack']

# Run Mann-Kendall test for each variable independently
mk_results = {}
for col in df_mk.columns:
    result = mk.original_test(df_mk[col])
    mk_results[col] = {
        'trend': result.trend,
        'h': result.h,
        'p': result.p,
        'Tau': result.Tau,
        's': result.s,
        'var_s': result.var_s,
        'slope': result.slope,
        'intercept': result.intercept
    }

# Display statistically significant results ordered by increasing p value
print("Statistically significant trends (p < 0.05):")
significant = [(var, res) for var, res in mk_results.items() if res['p'] < 0.05]
significant_sorted = sorted(significant, key=lambda x: x[1]['p'])
for var, res in significant_sorted:
    print(f"{var}: trend={res['trend']}, p={res['p']:.4g}, Tau={res['Tau']:.4f}")

print("\nStatistically insignificant trends (p >= 0.05):")
for var, res in mk_results.items():
    if res['p'] >= 0.05:
        print(f"{var}: trend={res['trend']}, p={res['p']:.4f}, Tau={res['Tau']:.4f}")

Statistically significant trends (p < 0.05):
votes_first: trend=decreasing, p=0, Tau=-0.1885
evo_length: trend=decreasing, p=0, Tau=-0.1623
num_abilities: trend=decreasing, p=8.57e-11, Tau=-0.1224
votes_top_6: trend=decreasing, p=7.65e-10, Tau=-0.1281
percent_male: trend=decreasing, p=1.613e-09, Tau=-0.1210
has_mega_evolution: trend=decreasing, p=8.304e-09, Tau=-0.0431
hp: trend=increasing, p=9.086e-08, Tau=0.1114
attack: trend=increasing, p=2.525e-07, Tau=0.1075
capture_rate: trend=decreasing, p=3.639e-07, Tau=-0.1044
defense: trend=increasing, p=9.936e-07, Tau=0.1020
against_normal: trend=decreasing, p=2.135e-06, Tau=-0.0673
against_fairy: trend=increasing, p=2.839e-06, Tau=0.0827
against_electric: trend=decreasing, p=0.000231, Tau=-0.0711
sp_defense: trend=increasing, p=0.0004775, Tau=0.0728
rarity: trend=increasing, p=0.0006425, Tau=0.0350
against_psychict: trend=decreasing, p=0.001051, Tau=-0.0576
sp_attack: trend=increasing, p=0.00222, Tau=0.0638
percent_female: trend=decreasing,

In [17]:
from scipy.stats import kruskal

# Exclude unwanted columns and 'gen', then add 'attack'
exclude = ['english_name', 'class_primary', 'class_secondary', 'gen']
variables = [col for col in df_clean.columns if col not in exclude]
if 'attack' not in variables:
    variables.append('attack')

# Group by generation
groups = [df_clean[df_clean['gen'] == gen] for gen in sorted(df_clean['gen'].unique())]

kruskal_results = {}
for var in variables:
    data = [group[var].values for group in groups]
    stat, p = kruskal(*data)
    kruskal_results[var] = {'statistic': stat, 'p_value': p}

# Show variables with significant differences across generations (p < 0.05), ordered by increasing p value
print("Variables with significant differences across generations (p < 0.05), ordered by increasing p value:")
significant_vars = [(var, res) for var, res in kruskal_results.items() if res['p_value'] < 0.05]
significant_vars_sorted = sorted(significant_vars, key=lambda x: x[1]['p_value'])
for var, res in significant_vars_sorted:
    print(f"{var}: statistic={res['statistic']:.2f}, p={res['p_value']:.4g}")


Variables with significant differences across generations (p < 0.05), ordered by increasing p value:
has_gigantamax: statistic=104.64, p=4.784e-19
votes_first: statistic=103.57, p=7.949e-19
base_egg_steps: statistic=93.12, p=1.081e-16
num_abilities: statistic=86.80, p=2.073e-15
percent_male: statistic=84.58, p=5.822e-15
has_mega_evolution: statistic=61.23, p=2.669e-10
rarity: statistic=60.85, p=3.175e-10
votes_top_6: statistic=60.08, p=4.49e-10
evo_length: statistic=52.96, p=1.1e-08
against_psychict: statistic=37.36, p=9.9e-06
capture_rate: statistic=35.42, p=2.241e-05
percent_female: statistic=33.98, p=4.105e-05
against_fairy: statistic=30.88, p=0.0001477
hp: statistic=29.34, p=0.0002761
attack: statistic=29.14, p=0.0002992
speed: statistic=21.33, p=0.006316
against_bug: statistic=20.22, p=0.009542
against_normal: statistic=18.16, p=0.02003
defense: statistic=16.62, p=0.03431


In [20]:
# For each possible division between generations (1 vs 2-9, 1-2 vs 3-9, ..., 1-8 vs 9)
# For each division, compare the two groups using several statistical tests for each variable

division_results = []

for split in range(1, 9):  # 8 possible splits between 9 generations
    # Define group indices
    gens_left = sorted(df_clean['gen'].unique())[:split]
    gens_right = sorted(df_clean['gen'].unique())[split:]
    
    # Split data
    left = df_clean[df_clean['gen'].isin(gens_left)]
    right = df_clean[df_clean['gen'].isin(gens_right)]
    
    stats_per_var = {}
    for var in variables:
        # Mann-Whitney U test (non-parametric)
        u_stat, u_p = stats.mannwhitneyu(left[var], right[var], alternative='two-sided')
        # t-test (parametric)
        t_stat, t_p = stats.ttest_ind(left[var], right[var], equal_var=False)
        # Kolmogorov-Smirnov test (distribution difference)
        ks_stat, ks_p = stats.ks_2samp(left[var], right[var])
        stats_per_var[var] = {
            'mannwhitneyu_p': u_p,
            'ttest_p': t_p,
            'ks_p': ks_p
        }
    
    # Multivariate tests: compare all variables at once
    left_matrix = left[variables].values
    right_matrix = right[variables].values
    # Hotelling's T2 test (using scipy.stats.ttest_ind for each variable, but not truly multivariate)
    # For a true multivariate test, use permutation MANOVA (e.g. via sklearn or pingouin)
    # Here, we use a simple approach: compare means and covariances
    mean_diff = np.linalg.norm(np.mean(left_matrix, axis=0) - np.mean(right_matrix, axis=0))
    cov_diff = np.linalg.norm(np.cov(left_matrix, rowvar=False) - np.cov(right_matrix, rowvar=False))
    # Permutation test for mean difference
    combined = np.vstack([left_matrix, right_matrix])
    n_left = left_matrix.shape[0]
    n_right = right_matrix.shape[0]
    n_perm = 1000
    perm_stats = []
    for _ in range(n_perm):
        idx = np.random.permutation(combined.shape[0])
        perm_left = combined[idx[:n_left], :]
        perm_right = combined[idx[n_left:], :]
        perm_stats.append(np.linalg.norm(np.mean(perm_left, axis=0) - np.mean(perm_right, axis=0)))
    p_perm = np.mean([s >= mean_diff for s in perm_stats])
    
    division_results.append({
        'split': split,
        'gens_left': gens_left,
        'gens_right': gens_right,
        'stats': stats_per_var,
        'mean_diff': mean_diff,
        'cov_diff': cov_diff,
        'perm_p': p_perm
    })

# Display summary: for each split, count variables with p < 0.05 for each test
for res in division_results:
    print(f"Division after generation {res['gens_left'][-1]} ({res['gens_left']} vs {res['gens_right']}):")
    for test in ['mannwhitneyu_p', 'ttest_p', 'ks_p']:
        sig_vars = [var for var, stat in res['stats'].items() if stat[test] < 0.05]
        print(f"  {test}: {len(sig_vars)} variables differ significantly (p < 0.05)")
    print()

Division after generation 1.0 ([1.0] vs [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]):
  mannwhitneyu_p: 16 variables differ significantly (p < 0.05)
  ttest_p: 23 variables differ significantly (p < 0.05)
  ks_p: 9 variables differ significantly (p < 0.05)

Division after generation 2.0 ([1.0, 2.0] vs [3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]):
  mannwhitneyu_p: 17 variables differ significantly (p < 0.05)
  ttest_p: 21 variables differ significantly (p < 0.05)
  ks_p: 11 variables differ significantly (p < 0.05)

Division after generation 3.0 ([1.0, 2.0, 3.0] vs [4.0, 5.0, 6.0, 7.0, 8.0, 9.0]):
  mannwhitneyu_p: 19 variables differ significantly (p < 0.05)
  ttest_p: 22 variables differ significantly (p < 0.05)
  ks_p: 14 variables differ significantly (p < 0.05)

Division after generation 4.0 ([1.0, 2.0, 3.0, 4.0] vs [5.0, 6.0, 7.0, 8.0, 9.0]):
  mannwhitneyu_p: 16 variables differ significantly (p < 0.05)
  ttest_p: 15 variables differ significantly (p < 0.05)
  ks_p: 14 variables differ sig